In [1]:
!pip install transformers==4.41.2
!pip install peft==0.10.0
!pip install accelerate
!pip install datasets
!pip install sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 

In [2]:
from huggingface_hub import login
login()

In [3]:
import torch
torch.cuda.empty_cache()

In [4]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [5]:
import pandas as pd

df = pd.read_csv("train.csv")

df.head()

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3


In [6]:
print(df.columns)

Index(['essay_id', 'full_text', 'score'], dtype='object')


In [7]:
df = df[['full_text', 'score']]

df = df.dropna()

df = df.sample(20)

In [8]:
def make_prompt(row):
    prompt = f"""
You are an expert essay grader.

Essay:
{row['full_text']}

Give a score from 1 to 6 only.
"""

    return {
        "text": prompt,
        "label": int(row['score'])
    }

data = df.apply(make_prompt, axis=1).tolist()

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [11]:
tokenizer.pad_token = tokenizer.eos_token

In [12]:

from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.10229075496156657


In [17]:
from datasets import Dataset

dataset = Dataset.from_list(data)

def tokenize(example):
    text = example["text"] + " Score: " + str(example["label"])

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(tokenize)

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    report_to="none",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4
)

In [19]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

In [20]:
trainer.train()

Step,Training Loss
10,2.728100
20,2.942100


TrainOutput(global_step=20, training_loss=2.835092067718506, metrics={'train_runtime': 4.2011, 'train_samples_per_second': 4.761, 'train_steps_per_second': 4.761, 'total_flos': 15907411722240.0, 'train_loss': 2.835092067718506, 'epoch': 1.0})

In [21]:
model.save_pretrained("aes-lora-model")
tokenizer.save_pretrained("aes-lora-model")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('aes-lora-model/tokenizer_config.json',
 'aes-lora-model/special_tokens_map.json',
 'aes-lora-model/tokenizer.model',
 'aes-lora-model/added_tokens.json',
 'aes-lora-model/tokenizer.json')

In [22]:
test_essay = """
Technology helps students learn more efficiently.
Online learning provides flexibility and access to information.
"""

prompt = f"""
You are an expert essay grader.

Essay:
{test_essay}

Give a score from 1 to 6 only.
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=20
)

print(tokenizer.decode(outputs[0]))

<s> 
You are an expert essay grader.

Essay:

Technology helps students learn more efficiently.
Online learning provides flexibility and access to information.


Give a score from 1 to 6 only.

1. Excellent
2. Good
3. Average
4. Poor
